In [1]:
"""
This script is to extract the imputed region matrix from pycistopic

authors: Roy Oelen
"""


'\nThis script is to extract the imputed region matrix from pycistopic\n\nauthors: Roy Oelen\n'

In [2]:
import os
import pycisTopic
import pandas as pd
import pickle
from scipy import sparse, io
from pycisTopic.diff_features import (
    impute_accessibility,
    normalize_scores,
    find_highly_variable_features,
    find_diff_features
)
import numpy as np

/home/umcg-roelen/miniconda3/envs/pycistopic_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2024-09-19 09:51:51,652	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [3]:
# location of the existing cistopic objects
cistopic_objects_loc='/groups/umcg-franke-scrna/tmp04/projects/multiome/ongoing/scenicplus_workdir/pycistopic/monocytes_all_cells/cistopic_objects/'
# the object to use
cistopic_object_loc=''.join([cistopic_objects_loc, 'cistopic_obj_20_topics_model.pkl'])
# read the object
with open(cistopic_object_loc, 'rb') as f:
    cistopic_object = pickle.load(f)


In [ ]:
# write the csr to an mtx file
mtx_loc = ''.join(['/groups/umcg-franke-scrna/tmp04/projects/multiome/ongoing/scenicplus_workdir/pycistopic/imputed_pycistopic_matrices/monocyte/', 'matrix.mtx'])
io.mmwrite(mtx_loc, cistopic_object.fragment_matrix)
# we will compress it post-hoc with bgzip /scratch/hb-functionalgenomics/projects/multiome/ongoing/scenicplus_workdir/pycistopic/imputed_pycistopic_matrices/monocyte/matrix.mtx

In [ ]:
# and write the cell names
cell_names = pd.DataFrame(data = {'barcode' : cistopic_object.cell_names})
# to a file
cell_names_loc = ''.join(['/groups/umcg-franke-scrna/tmp04/projects/multiome/ongoing/scenicplus_workdir/pycistopic/imputed_pycistopic_matrices/monocyte/', 'barcodes.tsv.gz'])
cell_names.to_csv(cell_names_loc, sep = '\t', header = False, index = False, compression = 'gzip')

In [ ]:
# and write the region names
feature_names = pd.DataFrame(data = {'barcode' : cistopic_object.region_names})
# to a file
feature_names_loc = ''.join(['/groups/umcg-franke-scrna/tmp04/multiome/ongoing/scenicplus_workdir/pycistopic/imputed_pycistopic_matrices/monocyte/', 'features.tsv.gz'])
feature_names.to_csv(feature_names_loc, sep = '\t', header = False, index = False, compression = 'gzip')

In [ ]:
# with finally the metadata as well
metadata = cistopic_object.cell_data
# to a file
metadata_loc = ''.join(['/groups/umcg-franke-scrna/tmp04/projects/multiome/ongoing/scenicplus_workdir/pycistopic/imputed_pycistopic_matrices/monocyte/', 'metadata.tsv.gz'])
metadata.to_csv(metadata_loc, sep = '\t', header = True, index = False, compression = 'gzip')

In [ ]:
# we will also export the region metadata (though we will probably not use it)
regiondata = cistopic_object.region_data
# to a file
regiondata_loc = ''.join(['/groups/umcg-franke-scrna/tmp04/projects/multiome/ongoing/scenicplus_workdir/pycistopic/imputed_pycistopic_matrices/monocyte/', 'regiondata.tsv.gz'])
regiondata.to_csv(regiondata_loc, sep = '\t', header = True, index = False, compression = 'gzip')

In [4]:
# impute accessibility
imputed_acc_obj = impute_accessibility(
    cistopic_object,
    selected_cells=None,
    selected_regions=None,
    scale_factor=10**6
)
del cistopic_object

2024-09-19 09:52:06,380 cisTopic     INFO     Imputing region accessibility
2024-09-19 09:52:06,381 cisTopic     INFO     Impute region accessibility for regions 0-20000
2024-09-19 09:52:18,509 cisTopic     INFO     Impute region accessibility for regions 20000-40000
2024-09-19 09:52:30,352 cisTopic     INFO     Impute region accessibility for regions 40000-60000
2024-09-19 09:52:42,205 cisTopic     INFO     Impute region accessibility for regions 60000-80000
2024-09-19 09:52:53,958 cisTopic     INFO     Impute region accessibility for regions 80000-100000
2024-09-19 09:53:05,768 cisTopic     INFO     Impute region accessibility for regions 100000-120000
2024-09-19 09:53:18,191 cisTopic     INFO     Impute region accessibility for regions 120000-140000
2024-09-19 09:53:30,084 cisTopic     INFO     Impute region accessibility for regions 140000-160000
2024-09-19 09:53:41,888 cisTopic     INFO     Impute region accessibility for regions 160000-180000
2024-09-19 09:53:53,806 cisTopic     

In [5]:
# write the output again, but now for the imputed data
mtx_loc = ''.join(['/groups/umcg-franke-scrna/tmp04/projects/multiome/ongoing/scenicplus_workdir/pycistopic/imputed_pycistopic_matrices/monocyte/', 'matrix.mtx'])
io.mmwrite(mtx_loc, sparse.csr_matrix(imputed_acc_obj.mtx))
cell_names = pd.DataFrame(data = {'barcode' : imputed_acc_obj.cell_names})
cell_names_loc = ''.join(['/groups/umcg-franke-scrna/tmp04/projects/multiome/ongoing/scenicplus_workdir/pycistopic/imputed_pycistopic_matrices/monocyte/', 'barcodes.tsv.gz'])
cell_names.to_csv(cell_names_loc, sep = '\t', header = False, index = False, compression = 'gzip')
feature_names = pd.DataFrame(data = {'barcode' : imputed_acc_obj.feature_names})
feature_names_loc = ''.join(['/groups/umcg-franke-scrna/tmp04/projects/multiome/ongoing/scenicplus_workdir/pycistopic/imputed_pycistopic_matrices/monocyte/', 'features.tsv.gz'])
feature_names.to_csv(feature_names_loc, sep = '\t', header = False, index = False, compression = 'gzip')